In [ ]:
import numpy as np
from numpy import sin , cos


x = np.array([1, 2])
y = np.array([2, 5])

matrix = np.array([x, y]).transpose()

vector: np.array = np.array([1, 2])

vector @ matrix


array([ 5, 12])

In [22]:
vector = np.array([1, 2])
print(vector )


[1 2]


## 生成正方体矩阵

In [37]:
np.array([[i, j, k] for i in [1,-1] for j in [1, -1] for k in [1, -1]]).transpose()

array([[ 1,  1,  1,  1, -1, -1, -1, -1],
       [ 1,  1, -1, -1,  1,  1, -1, -1],
       [ 1, -1,  1, -1,  1, -1,  1, -1]])

In [38]:
np.linspace(0, 2 * np.pi, 100)

array([0.        , 0.06346652, 0.12693304, 0.19039955, 0.25386607,
       0.31733259, 0.38079911, 0.44426563, 0.50773215, 0.57119866,
       0.63466518, 0.6981317 , 0.76159822, 0.82506474, 0.88853126,
       0.95199777, 1.01546429, 1.07893081, 1.14239733, 1.20586385,
       1.26933037, 1.33279688, 1.3962634 , 1.45972992, 1.52319644,
       1.58666296, 1.65012947, 1.71359599, 1.77706251, 1.84052903,
       1.90399555, 1.96746207, 2.03092858, 2.0943951 , 2.15786162,
       2.22132814, 2.28479466, 2.34826118, 2.41172769, 2.47519421,
       2.53866073, 2.60212725, 2.66559377, 2.72906028, 2.7925268 ,
       2.85599332, 2.91945984, 2.98292636, 3.04639288, 3.10985939,
       3.17332591, 3.23679243, 3.30025895, 3.36372547, 3.42719199,
       3.4906585 , 3.55412502, 3.61759154, 3.68105806, 3.74452458,
       3.8079911 , 3.87145761, 3.93492413, 3.99839065, 4.06185717,
       4.12532369, 4.1887902 , 4.25225672, 4.31572324, 4.37918976,
       4.44265628, 4.5061228 , 4.56958931, 4.63305583, 4.69652

In [ ]:
#定义旋转矩阵

def rotate_z(theta):
    col1 = [np.cos(theta), np.sin(theta), 0]
    col2 = [-np.sin(theta), np.cos(theta), 0]
    col3 = [0, 0, 1]
    return np.array([col1, col2, col3]).transpose()


def rotate_y(phi):
    col1 = [np.cos(phi), 0, -np.sin(phi)]
    col2 = [0, 1, 0]
    col3 = [np.sin(phi), 0, np.cos(phi)]
    return np.array([col1, col2, col3]).transpose()


#定义网格密度
U = np.linspace(0, 2 * np.pi, 100)
V = U

#定义图形半径
r = 0.3
R = 1

#定义图形坐标
donut = np.array([[np.cos(u) * (R + r * np.cos(v)), np.sin(u) * (R + r * np.cos(v)), r * np.sin(v)] for u in U for v in V]).transpose()

#定义旋转角度
phi = 1
theta = 2
transform = rotate_z(theta) @ rotate_y(phi)
donut_r = transform @ donut


#求面上的法向量
big_ring = R*np.array([[np.cos(u), (np.sin(u)), 0] for u in U for v in V]).transpose()
normal = donut - big_ring
normal = normal/r

#求所有法向量的范数
np.linalg.norm(normal, axis=0)

#构造打印显示矩阵
radius = 1.3 * (R + r)
res_x = 36
res_y = 18

x_line = np.linspace(-radius, radius, res_x)
y_line = np.linspace(-radius, radius, res_y)
pixils = np.stack(np.meshgrid(x_line, y_line), axis=2)

In [53]:
pixil_len_x = x_line[1] - x_line[0] 
pixil_len_y = y_line[1] - y_line[0]

def calculate_brightness_pixil(transform):
    screen = np.zeros([res_y, res_x])
    _donut = transform @ donut
    _normal = transform @ normal
    brightness_3d = _normal[2, :] + 1
    
    def in_pixil_window(points, pixil)->bool:
        x,y = pixil
        return(x-pixil_len_x < points[1, :]) & (points[1, :] < x+pixil_len_x) & (y-pixil_len_y < points[2, :]) & (points[2, :] < y+pixil_len_y)
    
    
    for i in range(res_y):
        for j in range(res_x):    
            mask = in_pixil_window(_donut , pixils[i][j]) 
            x_vals = _donut[0, :][mask]
            screen[i][j] = brightness_3d[mask][np.argmax(x_vals)] if x_vals.size > 0 else 0
    return screen

ASCII_BRIGHTNESS=".,-~:;=!*$$###@"

def show_donut(transform):
    screen = calculate_brightness_pixil(transform)
    scaled = np.zeros_like(screen)
    visible = screen > 0 
    if np.any(visible):
        lo = screen[visible].min()
        hi = screen[visible].max()
        scaled[visible] = (screen[visible] - lo) / (hi - lo)
    rows = []
    for row in scaled[::-1]:
        chars = []
        for value in row:
            if value <= 0:
                chars.append(" ")
            else:
                idx = min(len(ASCII_BRIGHTNESS)-1 , int(value * (len(ASCII_BRIGHTNESS)-1)))
                chars.append(ASCII_BRIGHTNESS[idx])
        rows.append("".join(chars))
    donut_frame = "\n".join(rows)
    return donut_frame

from IPython.display import clear_output
import time
angle = 0.0
while True:
    clear_output(wait=True)
    print(show_donut(rotate_z(angle) @ rotate_y(angle / 2)))
    angle += 0.1
    time.sleep(0.05)

                                    
                                    
          @##########               
       ######$$**!!*$$$#            
      ######$*==!=====!*$##         
     $#####$*!==!=====!!!*$##       
     $$$$$$$*!;~---~:;=!**$$$$      
     *$$$$$$$~    .,-:=!*$$$$$$     
     !!*$$$$$#       ~;!$$###$$     
     !!!****$$#       :$#####$$     
     :=!==!!!*$$##    ######$$!     
      :;==!=====**$$$###$$$$*!!     
       ~::;=====*=!=!==!!****!=     
         ,-~:;;===!=!==!!***;:      
           ..,-~~~::;;;;;;:~,       
                 ........           
                                    
                                    


                                    
                                    
                                    
       ####@@##                     
       $$########                   
       !*$*$$$#####                 
       !*=!=**$$$####               
        ;=!=*!!!**$$$#              
         ,:==!==!===!$$##           
           -~;==!===!=!*$#          
             ,~:;===!=!=!!*$        
                -~~::;;=!!*==       
                 ,,,--~:~;;=;       
                   ,..,,-~~:;       
                     , ...,,-       
                                    
                                    
                                    


KeyboardInterrupt: 